In [1]:
import time
import numpy as np
from matplotlib import pyplot as plt
from examples.seismic.utils import wiener_deconvolution, taper_wavelet, estimate_centroid_frequency_gather
from examples.seismic import SeismicModel, AcquisitionGeometry
from examples.seismic.vti import VTIWaveSolver
from examples.seismic.plotting import plot_two_wavelets, overlay_wiggle_plot
from examples.seismic.datasets import SeismogramDataset, VelocityModel
from devito import info
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
PATH_DATA = path = "../datasets/80_61_PreReady.sgy"
PATH_MODEL = "61_80_var1.dat"
SO = 4
N_GATHER = 100



# region Model definition
spacing = (0.075, 0.075)
velmodel = VelocityModel(PATH_MODEL, dx=spacing[0], dz=spacing[1])

origin = velmodel.x[0], velmodel.z[0]

vp = velmodel.vp.T * 1.03
epsilon = velmodel.epsilon.T
epsilon = velmodel.epsilon.T
delta = velmodel.epsilon.T

# velmodel.plot(show=True)

nbl = 500

model = SeismicModel(
vp=vp,
origin=origin,
shape=vp.shape,
spacing=spacing,
space_order=SO,
epsilon=epsilon,
delta=delta,
nbl=nbl,
bcs="damp",
vti=True,
fs=True,
)

vnx, vnz = model.grid.shape
vnx -= 2*nbl
vnz -= nbl

filename = f"snaps/snaps 101.bin"
nsnaps = 100    

fobj = open(filename, "rb")
snapsObj = np.fromfile(fobj, dtype=np.float32)
snapsObj = np.reshape(snapsObj, (nsnaps, vnx, vnz))
fobj.close()


qa = np.quantile(snapsObj, 0.98)
fig, ax = plt.subplots()
matrice = ax.imshow(snapsObj[0, :, :].T, vmin=-qa, vmax=qa, cmap="seismic")
plt.colorbar(matrice)

plt.xlabel('x')
plt.ylabel('z')

def update(i):
    matrice.set_array(snapsObj[i, :, :].T)
    return matrice,

# Animation
ani = animation.FuncAnimation(fig, update, frames=nsnaps, interval=50, blit=True)

plt.close(ani._fig)
HTML(ani.to_html5_video())


(2155, 553)


Operator `initdamp` ran in 0.01 s
